In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
UPSTASH_VECTOR_REST_READONLY_TOKEN=os.getenv('UPSTASH_VECTOR_REST_READONLY_TOKEN')
UPSTASH_VECTOR_REST_TOKEN=os.getenv('UPSTASH_VECTOR_REST_TOKEN')
UPSTASH_VECTOR_REST_URL=os.getenv('UPSTASH_VECTOR_REST_URL')

from upstash_vector import Index

index = Index(
    url=UPSTASH_VECTOR_REST_URL,
    token=UPSTASH_VECTOR_REST_TOKEN
    )

In [2]:
from upstash_vector import Index

index = Index(
    url=UPSTASH_VECTOR_REST_URL,
    token=UPSTASH_VECTOR_REST_TOKEN
    )

# index.upsert(
#   vectors=[
#       ("id1", "Enter data as string", {"metadata_field": "metadata_value"}),
#   ]
# )

# index.query(
#   data="Enter data as string",
#   filter="population >= 1000000 AND geography.continent = 'Asia'",
#   top_k=1,
#   include_vectors=True,
#   include_metadata=True
# )

In [32]:
from utility.property_listing_init import get_property_listing
import numpy as np

len(np.unique([x['slug'] for x in get_property_listing()]))

98

In [26]:
# preprocess.py

import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime, date

def make_json_serializable(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()

    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}

    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]

    return obj
HIGHWAY_EXPANSIONS = {
    r"\belite\b":           "ELITE Highway",
    r"\bskve\b":            "South Klang Valley Expressway SKVE",
    r"\bwce\b":             "West Coast Expressway WCE",
    r"\bsilk\b":            "Kajang Seremban Highway SILK",
    r"\bplus\b":            "North South Expressway PLUS",
    r"\bmex\b":             "Maju Expressway MEX",
    r"\bkesas\b":           "Kemuning Shah Alam Highway KESAS",
    r"\bnkve\b":            "New Klang Valley Expressway NKVE",
    r"\bspe\b":             "Setiawangsa Pantai Expressway SPE",
    r"\bduke\b":            "Duta Ulu Kelang Expressway DUKE",
    r"\bgce\b":             "Guthrie Corridor Expressway GCE",
    r"\bguthrie\b":         "Guthrie Corridor Expressway GCE",
    r"\bklia\b":            "Kuala Lumpur International Airport KLIA",
    r"\bmrt\b":             "Mass Rapid Transit MRT",
    r"\blrt\b":             "Light Rail Transit LRT",
    r"\bktm\b":             "Keretapi Tanah Melayu KTM commuter",
    r"\berl\b":             "Express Rail Link ERL",
    r"\bfederal highway\b": "Federal Highway Lebuhraya Persekutuan",
}

SECTION_HEADERS = re.compile(
    r"\b(property details|key features|location highlights|specifications?|"
    r"size breakdown|sizes?|connectivity|suitability|suitable for|features?|"
    r"highlights?|amenities|overview|building specs?)\s*[:\.]?",
    re.IGNORECASE,
)

# More surgical noise — avoid eating digits before/after
NOISE = re.compile(
    r"\(as stated\)|"
    r"\(please verify[^)]*\)|"
    r"\(note:[^)]*\)|"
    r"contact us[^.]*(?=\.|$)|"
    r"arrange a[^.]*viewing[^.]*(?=\.|$)|"
    r"there is a[^.]*discrepancy[^.]*(?=\.|$)|"
    r"description also (?:states|mentions)\s+\d[\d,]*\s*\w+|"  # match full "states 16,000 sqft"
    r"please verify[^,)]*",
    re.IGNORECASE,
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=25,
    separators=["\n\n", "\n", ". ", " "],
)


def clean(text: str) -> str:
    text = text.replace("\\n", "\n")
    text = re.sub(r"\n\s*[-•–]\s*", ". ", text)
    text = re.sub(r"\n+", ". ", text)
    text = SECTION_HEADERS.sub("", text)
    text = NOISE.sub("", text)

    # Collapse empty or whitespace-only parens left after noise removal
    text = re.sub(r"\(\s*[;,]?\s*\)", "", text)
    text = re.sub(r"\(\s*\)", "", text)

    for pattern, expansion in HIGHWAY_EXPANSIONS.items():
        if not re.search(re.escape(expansion), text, re.IGNORECASE):
            text = re.sub(pattern, expansion, text, flags=re.IGNORECASE)

    text = re.sub(r"\bapprox\.?\s*", "approximately ", text, flags=re.IGNORECASE)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r"\s([.,])", r"\1", text)

    # Strip any leading period/punctuation artifacts on the full text
    text = re.sub(r"^[\s.,;:]+", "", text)

    return text.strip()

def build_full_address(row: dict) -> str:
    location = row.get("location") or {}
    address  = location.get("address") or {}
    parts    = [
        location.get("industrial_park_name"),
        address.get("street_address"),
        address.get("address_locality"),
        address.get("address_region"),
        address.get("postal_code"),
        address.get("address_country"),
    ]
    return ", ".join(p for p in parts if p)

def build_combined(row: dict) -> str:
    # Use description only — features duplicate the same specs
    return clean(row["description"])

def build_full_address(row: dict) -> str:
    location = row.get("location") or {}
    address  = location.get("address") or {}
    parts    = [
        location.get("industrial_park_name"),
        address.get("street_address"),
        address.get("address_locality"),
        address.get("address_region"),
        address.get("postal_code"),
        address.get("address_country"),
    ]
    return ", ".join(str(p) for p in parts if p)

def build_metadata(row: dict, parent_text: str) -> dict:
    offer    = row.get("offer") or {}
    location = row.get("location") or {}
    address  = location.get("address") or {}

    return {
        # Identity
        "property_id":      row.get("property_id"),
        "slug":             row.get("slug"),
        "title":            row.get("title"),

        # Status
        "listing_status":   row.get("listing_status"),
        "market_status":    row.get("market_status"),
        "occupancy_status": row.get("occupancy_status"),

        # Offer
        "offer_type":       offer.get("offer_type"),
        "price":            offer.get("price"),
        "price_currency":   offer.get("price_currency"),
        "availability":     offer.get("availability"),

        # Category
        "main_category":    row.get("main_category"),
        "sub_categories":   row.get("sub_categories"),
        "tenure":           row.get("tenure"),

        # Location
        "full_address": build_full_address(row),
        "locality":     (row.get("location") or {}).get("address", {}).get("address_locality"),
        "region":       (row.get("location") or {}).get("address", {}).get("address_region"),

        # Sizes
        "land_sqft":        (row.get("land_size") or {}).get("value"),
        "built_up_sqft":    (row.get("built_up_area") or {}).get("value"),
        "office_area_sqft": (row.get("office_area") or {}).get("value"),

        # Specs
        "ceiling_height":       (row.get("ceiling_height") or {}).get("value"),
        "ceiling_height_unit":  (row.get("ceiling_height") or {}).get("unit"),
        "floor_loading":        (row.get("floor_loading") or {}).get("value"),
        "floor_loading_unit":   (row.get("floor_loading") or {}).get("unit"),
        "power_supply":         (row.get("power_supply") or {}).get("value"),
        "power_supply_unit":    (row.get("power_supply") or {}).get("unit"),

        # Construction
        "construction_status":   (row.get("construction") or {}).get("status"),
        "completion_year":       (row.get("construction") or {}).get("completion_year"),
        "completion_quarter":    (row.get("construction") or {}).get("completion_quarter"),

        # Dates
        "last_updated":     row.get("last_updated"),

        # Retrieval
        "parent_text":      parent_text,
    }


def preprocess(row: dict) -> dict:
    combined = build_combined(row)
    raw      = splitter.split_text(combined)
    chunks   = [
        re.sub(r"^[\s.,;:]+", "", c).strip()
        for c in raw
        if len(c.strip()) > 40
    ]

    metadata = build_metadata(row, parent_text=combined)

    return {
        "property_id": row.get("property_id"),
        "chunks":      chunks,
        "metadata":    metadata,
    }

def preprocess_batch(rows: list[dict]) -> list[dict]:
    return [preprocess(row) for row in rows]

In [27]:
def upsert(preprocessed: dict):
    property_id = preprocessed["property_id"]
    chunks      = preprocessed["chunks"]
    metadata = make_json_serializable(preprocessed["metadata"])


    vectors = [
        {
            "id":       f"{property_id}_chunk_{i}",
            "data":     chunk,
            "metadata": {
                **metadata,
                "chunk_index": i,
                "chunk_total": len(chunks),
            },
        }
        for i, chunk in enumerate(chunks)
    ]

    index.upsert(vectors=vectors)
    print(f"Upserted {len(vectors)} chunks for property_id={property_id}")


def upsert_batch(rows: list[dict]):
    preprocessed_rows = preprocess_batch(rows)
    for preprocessed in preprocessed_rows:
        upsert(preprocessed)

In [28]:
upsert_batch(get_property_listing())

Upserted 10 chunks for property_id=1
Upserted 19 chunks for property_id=2
Upserted 13 chunks for property_id=3
Upserted 14 chunks for property_id=4
Upserted 11 chunks for property_id=5
Upserted 12 chunks for property_id=6
Upserted 9 chunks for property_id=7
Upserted 9 chunks for property_id=8
Upserted 6 chunks for property_id=9
Upserted 9 chunks for property_id=10
Upserted 11 chunks for property_id=11
Upserted 15 chunks for property_id=12
Upserted 3 chunks for property_id=13
Upserted 11 chunks for property_id=14
Upserted 12 chunks for property_id=15
Upserted 9 chunks for property_id=16
Upserted 15 chunks for property_id=17
Upserted 13 chunks for property_id=18
Upserted 13 chunks for property_id=19
Upserted 17 chunks for property_id=20
Upserted 10 chunks for property_id=21
Upserted 11 chunks for property_id=22
Upserted 7 chunks for property_id=23
Upserted 10 chunks for property_id=24
Upserted 8 chunks for property_id=25
Upserted 8 chunks for property_id=26
Upserted 10 chunks for propert

In [13]:
index.query(
  data="factory",
#   filter="population >= 1000000 AND geography.continent = 'Asia'",
  top_k=30,
  include_metadata=True
)

[QueryResult(id='75_chunk_2', score=0.0305789, vector=None, metadata={'property_id': '75', 'slug': 'detached-factory-for-rent-telok-panglima-garang-kuala-langat-detached-factory', 'title': '82,000 sqft Detached Factory for Rent in Telok Panglima Garang', 'listing_status': 'active', 'market_status': None, 'occupancy_status': 'vacant', 'offer_type': 'rent', 'price': 139400, 'price_currency': 'MYR', 'availability': 'InStock', 'main_category': 'detached-factory', 'tenure': None, 'full_address': 'Taman Pertiwi, Taman Pertiwi, Teluk Panglima Garang, Kuala Langat, 42500, MY', 'locality': 'Teluk Panglima Garang, Kuala Langat', 'region': None, 'land_sqft': 3, 'built_up_sqft': 82000, 'office_area_sqft': 5000, 'ceiling_height': 13, 'ceiling_height_unit': 'm', 'floor_loading': 3, 'floor_loading_unit': 'ton/m²', 'power_supply': 1000, 'power_supply_unit': 'Amps', 'construction_status': None, 'completion_year': 2024, 'completion_quarter': None, 'last_updated': '2025-01-08T18:02:00Z', 'parent_text': '

In [10]:
def find_optimal_k(query: str, max_k: int = 50) -> list[dict]:
    results = index.query(
        data=query,
        top_k=max_k,
        include_metadata=True,
    )

    scores = [r.score for r in results]

    # Calculate drop between consecutive scores
    drops = [
        {
            "rank":       i + 1,
            "score":      scores[i],
            "next_score": scores[i + 1],
            "drop":       scores[i] - scores[i + 1],
            "drop_pct":   (scores[i] - scores[i + 1]) / scores[i] * 100,
            "property_id": results[i].metadata["property_id"],
        }
        for i in range(len(scores) - 1)
    ]

    # Find the biggest drop — that's your cutoff
    cutoff = max(drops, key=lambda x: x["drop"])

    print(f"Suggested cutoff: rank {cutoff['rank']} (drop of {cutoff['drop_pct']:.1f}%)\n")

    for d in drops:
        marker = " ← cutoff" if d["rank"] == cutoff["rank"] else ""
        print(f"rank={d['rank']:02d}  score={d['score']:.6f}  drop={d['drop_pct']:5.1f}%  pid={d['property_id']}{marker}")

    return drops

In [14]:
queries = [
    "factory near ELITE highway",
    "warehouse for rent in Balakong",
    "freehold semi-d factory high ceiling",
    "logistics facility near KLIA",
    "factory above 10000 sqft Selangor",
    "cold storage in klang"
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    find_optimal_k(q)



Query: factory near ELITE highway
Suggested cutoff: rank 22 (drop of 25.0%)

rank=01  score=0.030751  drop=  0.6%  pid=86
rank=02  score=0.030579  drop=  0.5%  pid=2
rank=03  score=0.030415  drop=  0.4%  pid=2
rank=04  score=0.030282  drop=  1.2%  pid=85
rank=05  score=0.029907  drop=  0.1%  pid=66
rank=06  score=0.029877  drop=  2.7%  pid=95
rank=07  score=0.029083  drop=  4.5%  pid=2
rank=08  score=0.027778  drop=  3.3%  pid=10
rank=09  score=0.026857  drop=  1.6%  pid=6
rank=10  score=0.026438  drop=  1.1%  pid=41
rank=11  score=0.026141  drop=  1.2%  pid=8
rank=12  score=0.025829  drop=  0.1%  pid=10
rank=13  score=0.025795  drop=  1.8%  pid=97
rank=14  score=0.025342  drop=  1.1%  pid=82
rank=15  score=0.025053  drop=  2.2%  pid=68
rank=16  score=0.024493  drop=  2.1%  pid=81
rank=17  score=0.023974  drop=  1.1%  pid=56
rank=18  score=0.023700  drop=  4.6%  pid=61
rank=19  score=0.022602  drop=  1.1%  pid=86
rank=20  score=0.022344  drop=  0.8%  pid=85
rank=21  score=0.022161  dr